# Multiple Progress Bars with Pythonic SSE

> Demonstrates running multiple concurrent tasks with independent progress tracking using a fully Pythonic approach. Each task has its own SSE connection, progress bar styling, and configuration - all generated from Python dataclasses without maintaining JavaScript strings.

In [1]:
from fasthtml.common import *
import uuid, time, random
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, List, Callable
from enum import Enum
import asyncio
from concurrent.futures import ThreadPoolExecutor

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_behaviors, btn_sizes
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.data_display.card import card, card_body, card_title
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors, badge_styles

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_align
from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import grid_display, gap, grid_cols, items, justify, flex_display
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.core.base import combine_classes

## Task Definitions with Different Characteristics

In [3]:
# Define various tasks with different speeds and behaviors
def data_processing_task():
    """Simulates data processing with consistent speed"""
    from tqdm import tqdm
    import time
    
    results = []
    for i in tqdm(range(150), desc="Processing data"):
        time.sleep(0.02)  # Consistent processing
        results.append(i ** 2)
    return {"status": "completed", "items_processed": len(results)}

def file_download_task():
    """Simulates file downloads with variable speed"""
    from tqdm import tqdm
    import time
    import random
    
    files = []
    for i in tqdm(range(80), desc="Downloading files"):
        # Variable download speed
        time.sleep(random.uniform(0.01, 0.05))
        files.append(f"file_{i}.dat")
    return {"status": "completed", "files_downloaded": len(files)}

def model_training_task():
    """Simulates ML model training with epochs"""
    from tqdm import tqdm
    import time
    
    losses = []
    for epoch in range(3):
        epoch_losses = []
        for batch in tqdm(range(50), desc=f"Epoch {epoch+1}/3"):
            time.sleep(0.03)
            loss = 1.0 / (batch + 1) * (3 - epoch)
            epoch_losses.append(loss)
        losses.append(sum(epoch_losses) / len(epoch_losses))
    return {"status": "completed", "final_loss": losses[-1]}

def image_processing_task():
    """Simulates image processing with batches"""
    from tqdm import tqdm
    import time
    
    processed = []
    for i in tqdm(range(100), desc="Processing images"):
        time.sleep(0.015)
        processed.append(f"img_{i:03d}_processed.jpg")
    return {"status": "completed", "images_processed": len(processed)}

def api_sync_task():
    """Simulates API synchronization with retries"""
    from tqdm import tqdm
    import time
    import random
    
    synced_records = []
    for i in tqdm(range(60), desc="Syncing with API"):
        # Simulate occasional retries
        if random.random() < 0.1:
            time.sleep(0.1)  # Retry delay
        else:
            time.sleep(0.02)
        synced_records.append({"id": i, "synced": True})
    return {"status": "completed", "records_synced": len(synced_records)}

## Enhanced Task Configuration System

In [4]:
@dataclass
class TaskDefinition:
    """Complete definition of a task with all its properties"""
    id: str
    name: str
    description: str
    function: Callable
    color_scheme: str = "primary"
    icon: str = "📊"  # Emoji icon for visual identification
    estimated_duration: float = 10.0  # seconds
    priority: int = 1
    auto_start: bool = False
    
@dataclass
class TaskUIConfig:
    """UI configuration for a specific task"""
    task_id: str
    show_eta: bool = True
    show_speed: bool = True
    decimal_places: int = 1
    show_percentage: bool = True
    show_counts: bool = True
    compact_mode: bool = False
    
@dataclass 
class TaskSSEConfig:
    """SSE configuration specific to a task"""
    task_id: str
    update_interval: float = 0.1
    heartbeat_interval: float = 10.0
    min_delta_pct: float = 1.0
    emit_initial: bool = True

## Multi-Task SSE Manager

In [5]:
class MultiTaskSSEManager:
    """Manages SSE connections for multiple tasks"""
    
    def __init__(self, tasks: List[TaskDefinition]):
        self.tasks = {task.id: task for task in tasks}
        self.ui_configs = {}
        self.sse_configs = {}
        
        # Create default configs for each task
        for task in tasks:
            self.ui_configs[task.id] = TaskUIConfig(task_id=task.id)
            self.sse_configs[task.id] = TaskSSEConfig(task_id=task.id)
    
    def generate_task_manager_script(self, task_id: str) -> str:
        """Generate SSE manager script for a specific task"""
        task = self.tasks[task_id]
        ui_config = self.ui_configs[task_id]
        sse_config = self.sse_configs[task_id]
        
        return f"""
window.taskManager_{task_id} = {{
    taskId: '{task_id}',
    taskName: '{task.name}',
    currentSource: null,
    startTime: null,
    
    config: {{
        endpoint: '/stream',
        updateInterval: {sse_config.update_interval * 1000},
        ui: {{
            containerId: '{task_id}-container',
            progressBar: '#{task_id}-progress',
            statusText: '#{task_id}-status',
            etaText: '#{task_id}-eta',
            speedText: '#{task_id}-speed',
            badge: '#{task_id}-badge',
            showEta: {'true' if ui_config.show_eta else 'false'},
            showSpeed: {'true' if ui_config.show_speed else 'false'},
            decimalPlaces: {ui_config.decimal_places},
            showPercentage: {'true' if ui_config.show_percentage else 'false'},
            showCounts: {'true' if ui_config.show_counts else 'false'}
        }}
    }},
    
    connect: function(jobId) {{
        if (this.currentSource) {{
            this.currentSource.close();
        }}
        
        this.startTime = Date.now();
        const url = this.config.endpoint + '?job_id=' + jobId;
        this.currentSource = new EventSource(url);
        
        this.currentSource.onmessage = this.handleMessage.bind(this);
        this.currentSource.onerror = this.handleError.bind(this);
        
        // Update badge to running
        this.updateBadge('running');
    }},
    
    handleMessage: function(event) {{
        const data = JSON.parse(event.data);
        
        // Update progress bar
        const progressBar = document.querySelector(this.config.ui.progressBar);
        if (progressBar) {{
            progressBar.value = data.progress;
        }}
        
        // Update status text
        this.updateStatus(data);
        
        // Update ETA if enabled
        if (this.config.ui.showEta) {{
            this.updateETA(data.progress);
        }}
        
        // Update speed if enabled
        if (this.config.ui.showSpeed && data.bars) {{
            this.updateSpeed(data.bars);
        }}
        
        // Handle completion
        if (data.completed) {{
            this.handleCompletion();
        }}
    }},
    
    updateStatus: function(data) {{
        const statusEl = document.querySelector(this.config.ui.statusText);
        if (!statusEl) return;
        
        if (data.bars && Object.keys(data.bars).length > 0) {{
            const bar = Object.values(data.bars)[0];
            let text = bar.desc + ': ';
            
            if (this.config.ui.showPercentage) {{
                text += bar.pct.toFixed(this.config.ui.decimalPlaces) + '%';
            }}
            
            if (this.config.ui.showCounts) {{
                text += ` (${{bar.cur}}/${{bar.tot}})`;
            }}
            
            statusEl.textContent = text;
        }} else {{
            statusEl.textContent = `Processing... ${{data.progress.toFixed(1)}}%`;
        }}
    }},
    
    updateETA: function(progress) {{
        const etaEl = document.querySelector(this.config.ui.etaText);
        if (!etaEl || progress === 0) return;
        
        const elapsed = (Date.now() - this.startTime) / 1000;
        const eta = (elapsed / progress) * (100 - progress);
        
        if (eta < 60) {{
            etaEl.textContent = `ETA: ${{Math.round(eta)}}s`;
        }} else {{
            etaEl.textContent = `ETA: ${{Math.round(eta / 60)}}m`;
        }}
    }},
    
    updateSpeed: function(bars) {{
        const speedEl = document.querySelector(this.config.ui.speedText);
        if (!speedEl || !bars) return;
        
        const bar = Object.values(bars)[0];
        if (bar.rate) {{
            speedEl.textContent = `Speed: ${{bar.rate.toFixed(1)}}/s`;
        }}
    }},
    
    updateBadge: function(status) {{
        const badge = document.querySelector(this.config.ui.badge);
        if (!badge) return;
        
        badge.textContent = status;
        badge.className = 'badge ' + (status === 'running' ? 'badge-warning' : 
                                     status === 'completed' ? 'badge-success' : 
                                     status === 'error' ? 'badge-error' : 'badge-ghost');
    }},
    
    handleCompletion: function() {{
        if (this.currentSource) {{
            this.currentSource.close();
            this.currentSource = null;
        }}
        
        // Update badge
        this.updateBadge('completed');
        
        // Update status
        const statusEl = document.querySelector(this.config.ui.statusText);
        if (statusEl) {{
            const elapsed = ((Date.now() - this.startTime) / 1000).toFixed(1);
            statusEl.textContent = `Completed in ${{elapsed}}s`;
        }}
        
        // Clear ETA
        const etaEl = document.querySelector(this.config.ui.etaText);
        if (etaEl) {{
            etaEl.textContent = '';
        }}
        
        // Re-enable start button
        const btn = document.querySelector(`#${{this.taskId}}-start-btn`);
        if (btn) {{
            btn.disabled = false;
            btn.classList.remove('btn-disabled');
        }}
    }},
    
    handleError: function(error) {{
        console.error(`SSE Error for task ${{this.taskId}}:`, error);
        this.updateBadge('error');
    }},
    
    disconnect: function() {{
        if (this.currentSource) {{
            this.currentSource.close();
            this.currentSource = null;
        }}
    }}
}};
"""
    
    def generate_all_scripts(self) -> List[Script]:
        """Generate scripts for all tasks"""
        return [Script(self.generate_task_manager_script(task_id)) 
                for task_id in self.tasks.keys()]

## Task Card Component Factory

In [6]:
class TaskCardFactory:
    """Factory for creating task card UI components"""
    
    def __init__(self, task: TaskDefinition, ui_config: TaskUIConfig):
        self.task = task
        self.ui_config = ui_config
    
    def create_task_card(self) -> Div:
        """Create a complete task card with all UI elements"""
        color_map = {
            "primary": progress_colors.primary,
            "secondary": progress_colors.secondary,
            "accent": progress_colors.accent,
            "success": progress_colors.success,
            "warning": progress_colors.warning,
            "info": progress_colors.info,
            "error": progress_colors.error
        }
        
        progress_color = color_map.get(self.task.color_scheme, progress_colors.primary)
        
        return Div(
            Div(
                # Card header
                Div(
                    Div(
                        Span(self.task.icon, cls=str(font_size._2xl)),
                        H3(self.task.name, cls=combine_classes(font_size.lg, font_weight.bold)),
                        cls=combine_classes(flex_display, items.center, gap(2))
                    ),
                    Span(
                        "idle",
                        id=f"{self.task.id}-badge",
                        cls=combine_classes(badge, badge_styles.ghost)
                    ),
                    cls=combine_classes(flex_display, justify.between, items.center, m.b(2))
                ),
                
                # Task description
                P(self.task.description, cls=combine_classes(font_size.sm, m.b(4))),
                
                # Progress bar
                Progress(
                    value="0",
                    max="100",
                    id=f"{self.task.id}-progress",
                    cls=combine_classes(progress, progress_color, w.full, m.b(2))
                ),
                
                # Status and metrics row
                Div(
                    P("Ready to start", id=f"{self.task.id}-status", cls=str(font_size.sm)),
                    Div(
                        Span("", id=f"{self.task.id}-speed", cls=combine_classes(font_size.xs, "text-gray-500")),
                        Span("", id=f"{self.task.id}-eta", cls=combine_classes(font_size.xs, "text-gray-500")),
                        cls=combine_classes(flex_display, gap(4))
                    ) if self.ui_config.show_speed or self.ui_config.show_eta else None,
                    cls=combine_classes(flex_display, justify.between, items.center, m.b(4))
                ),
                
                # Start button
                Button(
                    "Start Task",
                    id=f"{self.task.id}-start-btn",
                    hx_post=f"/start/{self.task.id}",
                    hx_target=f"#{self.task.id}-container",
                    hx_swap="none",
                    cls=combine_classes(
                        btn, 
                        getattr(btn_colors, self.task.color_scheme, btn_colors.primary),
                        btn_sizes.sm,
                        w.full
                    )
                ),
                
                cls=str(card_body)
            ),
            id=f"{self.task.id}-container",
            cls=combine_classes(card, "bg-base-100", "shadow-xl")
        )
    
    def create_start_script(self, job_id: str) -> Script:
        """Create script to start monitoring a task"""
        return Script(f"""
            // Disable button
            const btn = document.querySelector('#{self.task.id}-start-btn');
            if (btn) {{
                btn.disabled = true;
                btn.classList.add('btn-disabled');
            }}
            
            // Update status
            const status = document.querySelector('#{self.task.id}-status');
            if (status) {{
                status.textContent = 'Starting...';
            }}
            
            // Connect to SSE
            window.taskManager_{self.task.id}.connect('{job_id}');
        """)

## Dashboard Layout Manager

In [7]:
@dataclass
class DashboardConfig:
    """Configuration for the progress dashboard"""
    title: str = "Multi-Task Progress Dashboard"
    columns: int = 3  # Number of columns in grid
    show_master_controls: bool = True
    show_statistics: bool = True
    auto_refresh: bool = False
    refresh_interval: int = 5000  # milliseconds

class DashboardManager:
    """Manages the overall dashboard layout and controls"""
    
    def __init__(self, 
                 tasks: List[TaskDefinition],
                 config: DashboardConfig = None):
        self.tasks = tasks
        self.config = config or DashboardConfig()
        self.sse_manager = MultiTaskSSEManager(tasks)
        self.card_factories = {}
        
        for task in tasks:
            self.card_factories[task.id] = TaskCardFactory(
                task, 
                self.sse_manager.ui_configs[task.id]
            )
    
    def create_dashboard(self) -> Div:
        """Create the complete dashboard UI"""
        # Grid classes based on column count
        grid_cols_map = {
            1: "grid-cols-1",
            2: "grid-cols-1 md:grid-cols-2",
            3: "grid-cols-1 md:grid-cols-2 lg:grid-cols-3",
            4: "grid-cols-1 md:grid-cols-2 lg:grid-cols-4"
        }
        grid_cols_class = grid_cols_map.get(self.config.columns, "grid-cols-1 md:grid-cols-2 lg:grid-cols-3")
        
        components = []
        
        # Title
        components.append(
            H1(
                self.config.title,
                cls=combine_classes(
                    font_size._3xl,
                    font_weight.bold,
                    text_align.center,
                    m.b(6)
                )
            )
        )
        
        # Master controls
        if self.config.show_master_controls:
            components.append(self._create_master_controls())
        
        # Task cards grid
        task_cards = [factory.create_task_card() 
                     for factory in self.card_factories.values()]
        
        components.append(
            Div(
                *task_cards,
                cls=combine_classes(grid_display, grid_cols_class, gap(6))
            )
        )
        
        # Statistics panel
        if self.config.show_statistics:
            components.append(self._create_statistics_panel())
        
        return Div(
            *components,
            cls=str(p(8))
        )
    
    def _create_master_controls(self) -> Div:
        """Create master control buttons"""
        return Div(
            Button(
                "Start All Tasks",
                id="start-all-btn",
                hx_post="/start_all",
                hx_swap="none",
                cls=combine_classes(
                    btn,
                    btn_colors.success,
                    btn_sizes.md
                )
            ),
            Button(
                "Stop All Tasks",
                id="stop-all-btn",
                onclick="Object.keys(window).filter(k => k.startsWith('taskManager_')).forEach(k => window[k].disconnect());",
                cls=combine_classes(
                    btn,
                    btn_colors.error,
                    btn_sizes.md,
                    "ml-4"
                )
            ),
            cls=combine_classes(
                flex_display,
                justify.center,
                m.b(6)
            )
        )
    
    def _create_statistics_panel(self) -> Div:
        """Create statistics panel"""
        return Div(
            H2("Statistics", cls=combine_classes(font_size.xl, font_weight.bold, m.b(4))),
            Div(
                id="statistics-panel",
                cls=combine_classes(
                    "stats",
                    "shadow",
                    w.full,
                    m.t(6)
                )
            ),
            Script("""
                // Update statistics periodically
                setInterval(() => {
                    let running = 0, completed = 0, total = 0;
                    Object.keys(window).filter(k => k.startsWith('taskManager_')).forEach(k => {
                        total++;
                        if (window[k].currentSource) running++;
                        const badge = document.querySelector(`#${window[k].taskId}-badge`);
                        if (badge && badge.textContent === 'completed') completed++;
                    });
                    
                    const panel = document.getElementById('statistics-panel');
                    if (panel) {
                        panel.innerHTML = `
                            <div class="stat">
                                <div class="stat-title">Total Tasks</div>
                                <div class="stat-value">${total}</div>
                            </div>
                            <div class="stat">
                                <div class="stat-title">Running</div>
                                <div class="stat-value text-warning">${running}</div>
                            </div>
                            <div class="stat">
                                <div class="stat-title">Completed</div>
                                <div class="stat-value text-success">${completed}</div>
                            </div>
                        `;
                    }
                }, 1000);
            """)
        )
    
    def get_all_scripts(self) -> List[Script]:
        """Get all necessary scripts for the dashboard"""
        return self.sse_manager.generate_all_scripts()

## Application Setup

In [8]:
# Create app
app, rt = create_test_app(theme=DaisyUITheme.LIGHT)

# Initialize monitor and runner  
monitor = ProgressMonitor()
runner = JobRunner(monitor)

# Define available tasks
task_definitions = [
    TaskDefinition(
        id="data_proc",
        name="Data Processing",
        description="Processing large dataset with consistent speed",
        function=data_processing_task,
        color_scheme="primary",
        icon="📊",
        estimated_duration=3.0
    ),
    TaskDefinition(
        id="file_dl",
        name="File Downloads",
        description="Downloading files with variable network speed",
        function=file_download_task,
        color_scheme="info",
        icon="📥",
        estimated_duration=2.5
    ),
    TaskDefinition(
        id="ml_train",
        name="Model Training",
        description="Training ML model through multiple epochs",
        function=model_training_task,
        color_scheme="success",
        icon="🤖",
        estimated_duration=4.5
    ),
    TaskDefinition(
        id="img_proc",
        name="Image Processing",
        description="Batch processing images with filters",
        function=image_processing_task,
        color_scheme="warning",
        icon="🖼️",
        estimated_duration=1.5
    ),
    TaskDefinition(
        id="api_sync",
        name="API Sync",
        description="Synchronizing records with external API",
        function=api_sync_task,
        color_scheme="accent",
        icon="🔄",
        estimated_duration=2.0
    )
]

# Create dashboard manager
dashboard = DashboardManager(
    tasks=task_definitions,
    config=DashboardConfig(
        title="Multi-Task Progress Dashboard",
        columns=3,
        show_master_controls=True,
        show_statistics=True
    )
)

# Store active jobs
active_jobs = {}

## Routes

In [9]:
@rt("/")
def index():
    return create_test_page(
        "Multi-Task Progress Demo",
        dashboard.create_dashboard(),
        *dashboard.get_all_scripts()
    )

@rt("/start/{task_id}")
def start_task(task_id: str):
    """Start a specific task"""
    if task_id not in dashboard.tasks:
        return Div("Task not found", cls="text-error")
    
    job_id = str(uuid.uuid4())
    task = dashboard.tasks[task_id]
    
    # Start the job
    runner.start(
        job_id,
        task.function,
        patch_kwargs={
            "min_delta_pct": 1,
            "min_update_interval": 0.1,
            "emit_initial": True
        }
    )
    
    # Store job mapping
    active_jobs[task_id] = job_id
    
    # Return script to connect to SSE
    factory = dashboard.card_factories[task_id]
    return factory.create_start_script(job_id)

@rt("/start_all")
def start_all_tasks():
    """Start all tasks simultaneously"""
    scripts = []
    
    for task in dashboard.tasks:
        job_id = str(uuid.uuid4())
        
        # Start the job
        runner.start(
            job_id,
            task.function,
            patch_kwargs={
                "min_delta_pct": 1,
                "min_update_interval": 0.1,
                "emit_initial": True
            }
        )
        
        # Store job mapping
        active_jobs[task.id] = job_id
        
        # Add start script
        factory = dashboard.card_factories[task.id]
        scripts.append(factory.create_start_script(job_id))
    
    # Disable the start all button
    scripts.append(Script("""
        const btn = document.getElementById('start-all-btn');
        if (btn) {
            btn.disabled = true;
            btn.classList.add('btn-disabled');
            setTimeout(() => {
                btn.disabled = false;
                btn.classList.remove('btn-disabled');
            }, 2000);
        }
    """))
    
    return Div(*scripts)

@rt("/stream")
def stream(job_id: str):
    """SSE endpoint for streaming progress updates"""
    return EventStream(
        sse_stream(
            monitor,
            job_id,
            interval=0.1,
            heartbeat=10.0,
            wait_for_start=True,
            start_timeout=5.0
        )
    )

@rt("/status")
def get_status():
    """Get current status of all tasks"""
    status = {}
    for task_id, job_id in active_jobs.items():
        job_status = monitor.get_status(job_id)
        if job_status:
            status[task_id] = {
                "progress": job_status.get("progress", 0),
                "completed": job_status.get("completed", False),
                "bars": job_status.get("bars", {})
            }
    return status

## Start Server

In [10]:
# Start server for Jupyter
server = start_test_server(app)
display(HTMX())

In [11]:
# Stop server when done
server.stop()